# JapanDiv Pitch EDA: do index funds' dividend-reinvestment buys lift the market on the last cum-dividend day?

**The mechanism.** Most Japanese companies pay dividends to holders of record at the end of March and September. On
the ex-date the index drops by the dividend, but an index fund's NAV does not: the dividend becomes a receivable that
is only paid in June / December. To stay fully invested, Nikkei/TOPIX funds buy index futures equal to the expected
dividend at the **close of the last cum-dividend day** (Japanese press: 配当再投資の先物買い, about ¥1tn in March). That
is a known-date, mechanical, price-insensitive buy, and it grew with passive assets (BOJ / GPIF era, 2013–).

**Dates.** Record date = the month's last business day; last cum day = 3 business days before it (T+2 settlement
since 2019-07-16; 4 before that under T+3). Returns: close-to-close and open-to-close on the cum day, overnight into
the ex-date, and the 3 days after. Index returns on the ex-date include the dividend drop (price index), so ETFs
(1306.T TOPIX, 1321.T Nikkei) are shown too.

**Data.** Yahoo daily (`collect_prices.py`).

In [1]:
import pandas as pd, numpy as np
d=pd.read_csv("data/prices.csv",parse_dates=["date"])
O=d.pivot(index="date",columns="ticker",values="open"); C=d.pivot(index="date",columns="ticker",values="close")
out={}
for tk in ["^N225","1306.T","1321.T"]:
    n=C[tk].dropna(); o=O[tk].reindex(n.index); cal=n.index
    n=n[n>0]; cal=n.index; o=o.reindex(cal)
    cc=np.log(n).diff()*1e4; on=np.log(o/n.shift(1))*1e4; idr=np.log(n/o)*1e4
    rows=[]
    for y in range(1996,2027):
        for m in range(1,13):
            days=cal[(cal.year==y)&(cal.month==m)]
            if len(days)<5: continue
            k=2 if pd.Timestamp(y,m,1)>=pd.Timestamp("2019-07-16") else 3
            cum=days[-k-1]; i=cal.get_loc(cum)
            if i+5>=len(cal): continue
            rows.append({"tk":tk,"y":y,"m":m,"div":m in (3,9),"cum_cc":cc.iloc[i],"cum_id":idr.iloc[i],"ex_on":on.iloc[i+1],"ex_id":idr.iloc[i+1],"post3":cc.iloc[i+2:i+5].sum()})
    out[tk]=pd.DataFrame(rows)
for tk,R in out.items():
    for per,m in [("1996-2012",R.y<=2012),("2013-",R.y>=2013)]:
        d_=R[m&R["div"]]; o_=R[m&~R["div"]]
        print(tk,per,"Mar/Sep n",len(d_), {c:f"{d_[c].mean():+.0f} (t {d_[c].mean()/d_[c].sem():+.1f}) vs {o_[c].mean():+.0f}" for c in ["cum_cc","cum_id","ex_on","ex_id","post3"]})
R=out["^N225"]; R2=out["1321.T"]
print("\nN225 cum-day close-to-close by year, Mar / Sep:")
print(R[R["div"]&(R.y>=2008)].pivot(index="y",columns="m",values="cum_cc").round(0).T.to_string())

^N225 1996-2012 Mar/Sep n 34 {'cum_cc': '+9 (t +0.4) vs +12', 'cum_id': '+26 (t +1.6) vs +10', 'ex_on': '-12 (t -1.1) vs +6', 'ex_id': '+6 (t +0.3) vs -2', 'post3': '-40 (t -1.0) vs +39'}
^N225 2013- Mar/Sep n 28 {'cum_cc': '+57 (t +2.3) vs +7', 'cum_id': '+35 (t +2.2) vs -1', 'ex_on': '-74 (t -4.8) vs -3', 'ex_id': '+21 (t +1.4) vs +14', 'post3': '-80 (t -1.7) vs +17'}
1306.T 1996-2012 Mar/Sep n 8 {'cum_cc': '+112 (t +2.0) vs +4', 'cum_id': '+58 (t +2.8) vs -7', 'ex_on': '-17 (t -0.6) vs +11', 'ex_id': '+7 (t +0.3) vs +15', 'post3': '-77 (t -1.2) vs +17'}
1306.T 2013- Mar/Sep n 28 {'cum_cc': '+71 (t +3.1) vs +8', 'cum_id': '+39 (t +2.6) vs +1', 'ex_on': '-833 (t -1.0) vs +5', 'ex_id': '+26 (t +1.5) vs +18', 'post3': '+704 (t +0.8) vs -153'}
1321.T 1996-2012 Mar/Sep n 8 {'cum_cc': '+130 (t +2.1) vs +16', 'cum_id': '+inf (t +nan) vs +inf', 'ex_on': '-inf (t +nan) vs -inf', 'ex_id': '+inf (t +nan) vs +inf', 'post3': '-33 (t -0.5) vs +9'}
1321.T 2013- Mar/Sep n 28 {'cum_cc': '+41 (t +1.9)

/opt/miniconda3/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/miniconda3/lib/python3.13/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


**Interpretation.**
- **The cum-day pop exists since 2013:** Nikkei +57bp (t 2.3), TOPIX ETF +71bp (t 3.1), Nikkei ETF +41bp (t 1.9) on
  the last cum day of March/September, against +7–11bp in other months. Much of it is intraday (open → close +35–39bp),
  consistent with the close buy.
- **Before 2013 it is absent in the Nikkei** (+9bp, t 0.4), consistent with smaller passive assets, but that means
  there is no in-sample period to fit on and an out-of-sample period to test on. 2013– is simply where it exists.
- **It looks competed away recently.** The last four events (Mar/Sep 2025, Mar/Sep 2026) are all negative
  (−43 to −88bp). The run-up has moved earlier (pre-3-day +53bp) and a −80bp give-back follows over the next 3 days,
  the signature of front-running.
- 2 events a year: per-year Sharpe ≈ 0.6 at best on 28 events.

**Verdict: weak candidate, not promising enough.** The mechanism is real and visible, but the sample is small, the
split is not a true out-of-sample test, and the latest events reverse.